In [19]:
#!/usr/bin/python3 
import numpy as np
import dask.dataframe as dd
import scipy as sp
import xarray as xr
import pandas as pd
import dask.array as da
import sparse
import dask
import os
import itertools
from itertools import product
import pypardiso
import sys
from pstools.computation import map_blocks
import matplotlib.pyplot as plt
from scipy.sparse.linalg import cg

In [20]:
def reticolo(nomi_files, spacing, point = None, axes_names = None):
    '''
    il modulo prende i nomi dei file coi dati, la spaziatura, un eventuale punto di 
    ancoraggio (default = spaziatura) e eventuali nomi degli assi.
    Il punto di ancoraggio combacia col centro di un volumetto
    
    ritorna un dataarray che sintetizza tutte le proprietà
    '''
    #default
    if point == None:
        point = spacing

    if axes_names == None:
        axes_names = ["x", "y", "z"]

    #newcoords
    BB_rel = np.array([[-5, 5], [-5, 5], [-5, 5]])

    #inizializzazione
    nfiles = len(nomi_files)
    min_matr = da.zeros((nfiles, 3)) #le coordinate hanno 3 dimensioni
    max_matr = da.zeros((nfiles, 3))
    
    #leggo i file e trovo max e min per ogni file e per ogni dimensione
    for i in range(nfiles):
        ddf = dask.dataframe.read_csv(nomi_files[i]+"/*.part").repartition(partition_size='10MB')
        min_matr[i, :] = [ddf[axes_names].min().compute()]
        max_matr[i, :] = [ddf[axes_names].max().compute()]

    #minimo dei minimi e massimo dei massimi
    extremes = np.append(np.min(min_matr, 0), np.max(max_matr, 0))
    
    #raddoppio per usare un vettore solo
    double_spacing = np.append(spacing, spacing)
    double_point = np.append(point, point)
    
    #BB bounding box: volume di riferimento
    BB = np.transpose((np.floor(np.divide(extremes-double_point+double_spacing/2, double_spacing))*double_spacing + double_point).reshape(2, 3)) #2: min e max, 3: numero delle coordinate

    #axes = coordinate lungo i vari assi (centri dei volumetti)
    axes = [np.arange(BB[i, 0], BB[i, 1]+spacing[i], spacing[i]) for i in range(len(axes_names))]
    
    #bins = bordi di ogni cubetto
    bins = [np.append(axes[i][0]-spacing[i]/2, axes[i]+spacing[i]/2) for i in range(len(axes_names))]

    #for i in range(BB.shape[0]):
    #    if BB[i, 1]-BB[i, 0] == 0:
    #        print(i)
    #        BB_rel[i, 1] = BB_rel[i, 0] = ((BB[i, 0]-min(BB[:, 0]])*(max(BB_rel[:, 1])-min(BB_rel[:, 0]))/(max(BB[:, 1]-min(BB[:, 0]))+BB_rel[i, 0]).compute(). #????

    shape = list(i.size for i in axes)
    cube = xr.DataArray(name='cube', data=da.empty(shape, dtype='bool'),
                        dims=axes_names, coords={axes_names[i]: axes[i] for i in range(len(axes_names))})
    cube = cube.assign_attrs({'spacing': spacing, 'BB': BB, 'BB_rel': BB_rel, 'bins': bins, 'axes_names' :axes_names})  
    return cube

In [21]:
def points_counter(lungh_chunk, rank):
    '''
    rank: maschera dove 1 indica i punti validi;
    lungh_intervalli: lunghezza dei chunk nelle dimensioni
    '''
    #rank = da.from_array(r)
    x = rank.shape
    
    #divisione nei chunk
    bins = [np.arange(0, x[i]+lungh_chunk[i], lungh_chunk[i]) for i in np.arange(len(lungh_chunk))]
    r = sparse.as_coo(rank.compute()) #può essere migliorato

    #conto i punti nei vari chunk
    cont, _ = da.histogramdd(da.from_array(np.transpose(r.coords)), bins=bins)
    cont = cont.compute().reshape((1, np.prod(cont.shape)))[0]
    
    #costruinsco una matrice triangolare inferiore e conto lungo le colonne
    cont = cont[:, None]*np.ones(len(cont))[None, :]
    cont = np.triu(cont)
    return np.append(0, np.sum(cont, 0))

In [22]:
def simulo_sensore(n_punti, nomefile, LOS_mean, axes_names=["x", "y", "z"], LOS_axes = None):
    '''
    il modulo prende in input numero di punti, un nome di 
    file csv nel quale vengono scritti i dati simulati, un vettore (LOS a meno di fluttuazioni random),
    il nome degli assi da usare e il nome degli assi lungo cui calcolare la LOS
    '''

    LOS_ndims = len(LOS_mean)
    
    #inizilizzazione
    if LOS_axes == None:
        LOS_axes = axes_names[0:LOS_ndims]
        
    #simulo le coordinate (sempre in 3 dimensioni)
    punti = 100*da.random.random_sample((n_punti, 3))
    
    #simulo le LOS data la LOS media
    LOS = LOS_mean[None, :]*da.ones(n_punti)[:, None] + da.random.random_sample((n_punti, LOS_ndims))
        
    #normalizzo
    L = da.sqrt(da.sum(da.square(LOS), axis = 1))
    LOS = LOS/(L[:, None]*da.ones(LOS_ndims)[None, :])
    
    #nomi alle LOS
    LOS_names = ["LOS_{}".format(i) for i in LOS_axes]
        
    #simulo i termini noti
    misure = 20*da.random.random_sample(n_punti)
    
    #creo il dataframe con il dizionario assi:misure
    d1 = {axes_names[0]: punti[:, 0], axes_names[1]: punti[:, 1], axes_names[2]: punti[:, 2], 'misure': misure}
    d2 = {LOS_names[i]: LOS[:, i] for i in range(LOS_ndims)}
    
    df = pd.DataFrame({**d1, **d2})

    ddf = dask.dataframe.from_pandas(df).repartition(partition_size='10MB')
    if os.path.isdir(nomefile):
        for file in os.listdir(nomefile):
            os.remove(nomefile+'/'+file)
        os.removedirs(nomefile)
    
    #scrivo il file
    ddf.to_csv(nomefile, mode='x')
    
    return 

In [23]:
def clusterizzazione(filename, cube, grado):
    ''' 
    questo modulo approssima le coordinate coi centri dei volumetti  
    per ogni sensore e media le rilevazioni che rientrano nello stesso volumetto
    '''
    #nomi delle coordinate
    axes_names = cube.attrs.get('axes_names')       

    #leggo file
    ddf = dask.dataframe.read_csv(filename+"/*.part").repartition(partition_size='10MB')
    
    #leggo le coordinate
    #punti = da.transpose(da.from_array([ddf[axes_names[0]], ddf[axes_names[1]], ddf[axes_names[2]]]))
    punti = ddf[axes_names].to_dask_array(lengths = True)
    #print(punti.compute())
    #punti = np.array([(punti[:, i]-BB[i, 0])*(BB_rel[i, 1]-BB_rel[i, 0])/(BB[i, 1]-BB[i, 0])+BB_rel[i, 0] for i in range(punti.shape[1])])
    #punti = np.array([(punti[:, i]-BB[i, 0])*(BB_rel[i, 1]-BB_rel[i, 0]) for i in range(punti.shape[1])])
    #print("shape punti")
    #print(punti)
    #dist = np.array([BB[i, 1]-BB[i, 0] for i in range(punti.shape[0])])[:, None]*np.ones(punti.shape[1])[None, :]
    #dist[np.where(dist==0)]=1
    #print(dist)

    #punti = punti/dist
    #punti = np.array([punti[i, :]+BB_rel[i, 0] for i in range(punti.shape[0])])
    #print(punti)
    #punti = da.from_array(punti)
    #leggo come dividere le coordinate (bins: separazioni tra i cubetti, coordinate: centro dei cubetti)
    B = cube.attrs.get('bins')
    B = [np.array(B[i]) for i in range(len(B))]
    #print(B.shape)
    #print(punti)
    #punti=punti.transpose()
    #p = [punti[i, :] for i in range(punti.shape[0])]
    #len_ints = cube.attrs.get("spacing")
    #coordinate = cube.attrs.get('axes')
    coordinate = [cube.coords[name] for name in axes_names]
    
    #leggo la los coi suoi assi
    #LOS = da.from_array(ddf.loc[:, ddf.columns.str.startswith('LOS')].compute().to_numpy()) #passo per pandas
    #LOS = da.from_array(ddf.loc[:, ddf.columns.str.startswith('LOS')])
    LOS_axes = list(ddf.loc[:, ddf.columns.str.startswith('LOS')].columns)
    LOS = ddf[LOS_axes].to_dask_array(lengths=True)

    LOS_ndims = len(LOS_axes)
    
    #print(punti.shape)
    #print(B.shape)
    #conto i punti per ogni cubetto
    denominator, _ = da.histogramdd(punti, bins=B) 
    
    #prod: matrice con una riga di 0, una riga di 1 (e una di 2)
    vec = np.ones(LOS_ndims)
    prod = punti[:, None]*vec[:, None]
    
    #vettore con le LOS: prima tutte le x, poi tutte le y, poi tutte le z
    #LL = np.transpose(LOS).reshape(punti.shape[0]*LOS_ndims)
    LL = da.transpose(LOS).reshape(punti.shape[0]*LOS_ndims)#np->da
    
    #matrice co una riga di 0, una di 1 e una di 2
    vec2 = np.arange(LOS_ndims)[:, None]*np.ones((1, punti.shape[0]))

    #A sono le coordinate delle LOS: in ogni riga (per ogni valore delle LOS) abbiamo le 3 coordinate più un numero che indica a quale 
    # asse (quale coordinata delle LOS rappresenta) appartiene il valore
    #A = np.zeros((punti.shape[0]*LOS_ndims, 4)) #4: 3 coordinate dei punti + asse, punti.shape[0]*LOS_ndims "sciolgo" i punti
    A = da.zeros((punti.shape[0]*LOS_ndims, 4))
    A[:,np.arange(3)] = prod.transpose(1, 0, 2).reshape((punti.shape[0]*LOS_ndims, 3))
    A[:, 3]=vec2.reshape(punti.shape[0]*LOS_ndims) #n_punti * 0 + n_punti*1+ (n_punti*2)
    
    #C sono i bins per A che ha 4 dimensioni: ma la 4 può assumere solo i valori discreti 0, 1, (2)
    C = B.copy()
    C.append(np.array([-0.5, 0.5, 1.5, 2.5])[np.arange(LOS_ndims + 1)])
    LL = da.rechunk(LL, A.chunks)

    #sommo i valori delle LOS per ogni punto e ogni coordinata delle LOS in esame
    #L, _ = da.histogramdd(da.from_array(A), bins=C, weights= LL)
    L, _ = da.histogramdd(A, bins=C, weights= LL)
    
    #denominatore in 4 dimensioni
    den = denominator[:, :, :, None]*vec[None, :]
    
    #divido per il numero di punti per ottenere la media
    L /= den

    #sommo e medio i termini noti
    #misure = da.from_array([ddf['misure']]).flatten()

    misure = ddf['misure'].to_dask_array(lengths = True)
    misure = da.rechunk(misure, punti.chunks)

    mis, _ = da.histogramdd(punti, bins=B, weights=misure)
    mis /= denominator

    #creo il dataset
    data_set = xr.Dataset({'LOS': ([axes_names[0], axes_names[1], axes_names[2], 'axes'], L),
                           'mis': (axes_names, mis)},
                          coords = {axes_names[0]: coordinate[0], axes_names[1]: coordinate[1], axes_names[2]: coordinate[2], 'axes' : LOS_axes})
    data_set =data_set.assign_attrs(LOS_media= np.mean(LOS.compute(), 0))

    #assegno il grado del polinomio (preso in input) con cui approssimare l'incertezza spaziale
    data_set = data_set.assign_attrs(grado = grado)
    
    return data_set

In [24]:
def set_pol(grado):
    '''
    Questo modulo ritorna una matrice dove ogni riga contiene i coefficienti a cui 
    elevare le coordinate del punto.
    Il modulo ritorna tutte le combinazioni di 3 elementi con gli elementi minori 
    o uguali al grado passato in input con somma minore o uguale al grado stesso.
    '''
    
    #lista di tutte le combinazioni di 3 elementi di tutti i numeri minori o uguali del grado
    l = list(set(product(set(range(grado + 1)),repeat = 3))) #3:x, y, z
    l.sort()
    
    #trasformo la lista in un array
    matr = np.array(l)
    
    #seleziono solo le righe con somma minore del grado passato in input
    return matr[np.sum(matr, 1)<=grado, :]

In [45]:
def funzione_spostamento(punti):
    
    return np.array([punti[:, 0]//5-2*punti[:, 1]%7, punti[:, 2]//3+6*punti[:, 0]//7-10%punti[:, 1], punti[:, 2]//4])

In [26]:
def polynomial(coordinates, grade):
    '''
    Questo modulo crea una matrice dove ogni riga ha i vari elementi del polinomio del grado
    passato in input 
    '''
    
    #polinomio con gli esponenti
    matr = set_pol(grade)
    
    #applico l'elevamento a potenza
    return np.multiply(np.multiply((np.array(coordinates[:, 0])[:, None]** 
                                    np.array(matr[:, 0])[None, :]), (np.array(coordinates[:, 1])
                                    [:, None]** np.array(matr[:, 1])[None, :])), 
                       (np.array(coordinates[:, 2])[:, None]** np.array(matr[:, 2])
                        [None, :])).reshape((coordinates.shape[0], matr.shape[0]))

In [ ]:
def simulo_sensore_da_soluzione(n_punti, nomefile, LOS_mean, grado, rumore=0, axes_names=["x", "y", "z"], LOS_axes = None):

    #simulo le coordinate (sempre in 3 dimensioni)
    punti = 100*da.random.random_sample((n_punti, 3))
    LOS_ndims = len(LOS_mean)
    
    #inizilizzazione
    if LOS_axes == None:
        LOS_axes = axes_names[0:LOS_ndims]
        
    #simulo le LOS data la LOS media
    LOS = LOS_mean[None, :]*da.ones(n_punti)[:, None] + da.random.random_sample((n_punti, LOS_ndims))
        
    #normalizzo
    L = da.sqrt(da.sum(da.square(LOS), axis = 1))
    LOS = LOS/(L[:, None]*da.ones(LOS_ndims)[None, :])
    
    #nomi alle LOS
    LOS_names = ["LOS_{}".format(i) for i in LOS_axes]
    
    spostamento = funzione_spostamento(punti)
    pol = polynomial(punti, grado)
    coeff = 10*np.random.rand(pol.shape[1])#coefficienti reali?
    print(pol@coeff)
    print()
    
    #creo il dataframe con il dizionario assi:misure
    d1 = {axes_names[0]: punti[:, 0], axes_names[1]: punti[:, 1], axes_names[2]: punti[:, 2]}
    d2 = {LOS_names[i]: LOS[:, i] for i in range(LOS_ndims)}
    
    df = pd.DataFrame({**d1, **d2})

    ddf = dask.dataframe.from_pandas(df).repartition(partition_size='10MB')
    if os.path.isdir(nomefile):
        for file in os.listdir(nomefile):
            os.remove(nomefile+'/'+file)
        os.removedirs(nomefile)
    
    #scrivo il file
    ddf.to_csv(nomefile, mode='x')
    
    return 

In [28]:
def sens_counter(lista_sens):
    '''
    Il modulo conta quanti coefficienti servono per stimare il polinomio che rappresenta il 
    filtro spaziale per ogni sensore. 
    Il modulo ritorna un vettore la cui i-esima posizione è occupata dalla somma dei coefficienti 
    fino all'(i-1)esimo sensore. Il vettore è lungo (numero sensori + 1).
    '''
    #il primo numero è 0
    vec = [0]
    
    #riempio il vettore
    for i in range(len(lista_sens)): 
        n = len(set_pol(lista_sens[i].attrs.get('grado')))
        vec.append(vec[len(vec)-1]+n)
    return vec

In [29]:
def chunk_matrices(num_chunk, intervallo_chunk, lista_sensori, maschera, cubo, interesting_dimensions, conto_punti):
    '''
    Questo modulo scrive le porzioni di matrici e termine noto per un pezzo di dominio. Gli input sono:
    num_chunk: il numero del chunk, identificativo della porzione di dominio da analizzare;
    - intervallo_chunk: quanti sottointervalli sono compresi nella porzione di dominio (per tagliare
    - il dominio, non necessario alla ostruzione della matrice di per sé);
    - lista_sensori: lista dei dataset coi dati raccolti da ogni sensore;
    - maschera: maschera che assegna a ogni punto 1 se è valido, 0 altrimenti;
    - cubo: dataset con le coordinate del dominio e le informazioni;
    - interesting_dimensions: il numero di dimensioni delle LOS da considerare;
    - conto_punti: vettore la cui i_esima posizione è occupata dal numero di punti validi nei chunk dal primo all'(i-1)esimo.
    
    IDEA: 
    per ogni chunk costruisco una striscia della matrice totale e un pezzo del termine noto. Come vengono assemblate le strisce 
    non è rilevante, l'importante è che anche il termine noto venga assemblato con lo stesso ordine. La striscia di matrice si 
    compone di due matrici: A, che contiene i coefficienti delle LOS, e B, che contiene i polinomi con le coordinate. 
    I dati vengono esaminati un lettore alla volta: la matrice B sarà quindi diagonale a blocchi, mentre la A è a "strisce".
    '''
    
    ##GRATIS
    
    #calcolo il numero di chunk in ogni dimensione e estrapolo le coordinate dell'inizio del blocco
    Nx, Ny, Nz = maschera.shape//intervallo_chunk + np.ones(3)*(maschera.shape%intervallo_chunk>0)
    coords = [int(num_chunk//(Nz*Ny)),  int(num_chunk//Nz%Ny), int(num_chunk%Nz)]
    
    #calcolo una volta per tutte
    axes = cubo.attrs.get("axes")
    num_sensori = len(lista_sensori)
    
    #taglio le coordinate nel chunk: i centri dei cibetti coinvolti nel chunk
    intervals = [axes[i][coords[i]*intervallo_chunk[i]:min((coords[i]+1)*intervallo_chunk[i], len(axes[i]))] for i in range(3)]
    
    #taglio gli elementi delle liste in input
    maschera = maschera.compute()[coords[0]*intervallo_chunk[0]:(coords[0]+1)*intervallo_chunk[0], coords[1]*intervallo_chunk[1]:(coords[1]+1)*intervallo_chunk[1], 
                                        coords[2]*intervallo_chunk[2]:(coords[2]+1)*intervallo_chunk[2]]
    lista_LOS_sens_ridotta = [sens.LOS[coords[0]*intervallo_chunk[0]:(coords[0]+1)*intervallo_chunk[0], coords[1]*intervallo_chunk[1]:(coords[1]+1)*intervallo_chunk[1], 
                                        coords[2]*intervallo_chunk[2]:(coords[2]+1)*intervallo_chunk[2]] for sens in lista_sensori]
    lista_mis_sens_ridotta = [sens.mis[coords[0]*intervallo_chunk[0]:(coords[0]+1)*intervallo_chunk[0], coords[1]*intervallo_chunk[1]:(coords[1]+1)*intervallo_chunk[1], 
                                        coords[2]*intervallo_chunk[2]:(coords[2]+1)*intervallo_chunk[2]] for sens in lista_sensori]
    ##FINE PARTE GRATUITA

    #per la colonna
    valid_points = np.count_nonzero(maschera) #numero dei punti validi nel chunk

    num_points_total = int(conto_punti[len(conto_punti)-1]) #punti totali

    #per inizializzare la matrice
    sens_count = sens_counter(lista_sensori) 
    
    #inizializzo la matrice che comprende tutti i polinomi con le coordinate: mi serve sapere il numero di colonne totale, 
    # ovvero quanto spazio verrà occupato in totale da tutti i sensori. Il numero di colonne è stimato per eccesso
    B = sp.sparse.dok_matrix(np.zeros((num_sensori*valid_points, sens_count[len(sens_count)-1])))
    
    #check
    if valid_points == 0:
        print("no valid points in chunk {}".format(num_chunk))
        #inizializzo una matrice a nulla e un termine noto nullo di modo che la composizione finale si possa comunque fare
        a = sp.sparse.csr_matrix((0, num_points_total*interesting_dimensions))
        mis = np.array([])
        return mis, a, B

    #coordinate dei punti validi
    block_coords = np.where(maschera>0)
    
    #matrice sparsa dove a ogni punto valido è associato un numero naturale 
    sp_mat = sparse.COO([block_coords[0], block_coords[1], block_coords[2]], np.arange(valid_points)+1, fill_value=-1)

    #inizializzazione
    misure = []
    a_list = []
    counter = 0
    print("a")
    for i in np.arange(len(lista_LOS_sens_ridotta)): #per ogni sensore
        coordinates = np.where(da.isfinite(lista_mis_sens_ridotta[i][block_coords]) == True)#NO
        
        #check: entro solo se il sensore ha i dati di almeno un punto valido
        #if np.isfinite(lista_mis_sens_ridotta[i].data.any()):
        #if da.isfinite(lista_mis_sens_ridotta[i][block_coords].all()):
        if len(coordinates[0])>0:
            #intersezione tra i punti validi e i punti di cui il sensore ha rilevazioni
            #coordinates = np.where((da.isfinite(lista_mis_sens_ridotta[i].values) == True) & (maschera>0))

            #indice relativo dei punti validi analizzati dal sensore
            list_points = sp_mat[coordinates[0], coordinates[1], coordinates[2]]-1
            list_points = list_points.todense()
            
            #grado del polinomio che funge da filtro spaziale
            grado = lista_sensori[i].grado
            
            #coordinate assolute dei punti validi analizzati dal sensore
            absolute_coordinates = np.array([intervals[0][coordinates[0]], intervals[1][coordinates[1]], intervals[2][coordinates[2]]])

            #numero di punti validi nel sensore
            n_points = len(coordinates[0])

            #creo una striscia di A relativa al sensore: uso tante righe quante il numero di punti validi nel sensore e per ogni riga 
            #occupo 3 colonne. l'i_esimo punto valido relativo a questo sensore occuperà la riga i alle colonne (numero di punti nei 
            # chunk precedenti a questo + indice relativo del sensore), e il o i successivi in base a quante sono le componenti 
            #della LOS a cui siamo interessati, che saranno i dati che andremo a piazzare
            rows_a = np.arange(n_points)[:, None]*np.ones(interesting_dimensions)[None, :]
            rows_a = rows_a.reshape((n_points*interesting_dimensions, ))

            columns_a = int(conto_punti[num_chunk])*interesting_dimensions+list_points[:, None]*interesting_dimensions+ np.arange(interesting_dimensions)[None, :]
            columns_a = columns_a.reshape((n_points*interesting_dimensions, ))

            data_a = lista_LOS_sens_ridotta[i].values[coordinates[0], coordinates[1], coordinates[2], :].reshape(n_points*interesting_dimensions)

            a = sp.sparse.coo_array((data_a, (np.array([rows_a, columns_a]))), shape = (n_points, interesting_dimensions*num_points_total))
            a_list.append(a)
            
            #inserisco i valori in B
            B[counter:counter+n_points, sens_count[i]:sens_count[i+1]] = sp.sparse.coo_array(polynomial(np.transpose(absolute_coordinates), grado))

            #il termine noto comprende le misurazioni
            misure.append(list(lista_mis_sens_ridotta[i].values[coordinates]))
                        
            #il counter serve per sapere quali righe di B sono effettivamente usate
            counter += n_points
         
    #assemblo A e il termine noto
    mis = np.hstack(misure).flatten()
    A = sp.sparse.vstack([a for a in a_list])

    #taglio B
    B=B[0:counter, :]

    return mis, A, B

In [30]:
def chunk_matrices_tagliata(cubo, maschera, *lista_sensori, job_id=0):
    '''
    Questo modulo scrive le porzioni di matrici e termine noto per un pezzo di dominio. Gli input sono:
    - lista_sensori: lista dei dataset coi dati raccolti da ogni sensore;
    - maschera: maschera che assegna a ogni punto 1 se è valido, 0 altrimenti;
    - cubo: dataset con le coordinate del dominio e le informazioni;
    
    Dal cubo prendo le seguenti indicazioni:
    - interesting_dimensions: il numero di dimensioni delle LOS da considerare;
    - conto_punti: vettore la cui i_esima posizione è occupata dal numero di punti validi nei chunk dal primo all'(i-1)esimo.
    
    IDEA: 
    per ogni chunk costruisco una striscia della matrice totale e un pezzo del termine noto. Come vengono assemblate le strisce 
    non è rilevante, l'importante è che anche il termine noto venga assemblato con lo stesso ordine. La striscia di matrice si 
    compone di due matrici: A, che contiene i coefficienti delle LOS, e B, che contiene i polinomi con le coordinate. 
    I dati vengono esaminati un lettore alla volta: la matrice B sarà quindi diagonale a blocchi, mentre la A è a "strisce".
    '''
    #prendo dal cubo le informazioni
    interesting_dimensions = cubo.attrs.get("interesting_dimensions")
    conto_punti = cubo.attrs.get("conto_punti")
    
    #identificativo del chunk che sto analizzando (richiesto da map_blocks)
    num_chunk = job_id
    
    #per la colonna
    valid_points = da.count_nonzero(maschera) #numero dei punti validi nel chunk
    
    #per inizializzare la matrice
    sens_count = sens_counter(lista_sensori)
    num_points_total = int(conto_punti[len(conto_punti)-1]) #punti totali
    
    BB = cubo.attrs.get("BB")
    BB_rel = cubo.attrs.get("BB_rel")
    
    #check
    if valid_points == 0:
    #if int(conto_punti[num_chunk+1])-int(conto_punti[num_chunk])==0:
        print("no valid points in chunk {}".format(num_chunk))
        #inizializzo una matrice a nulla e un termine noto nullo di modo che la composizione finale si possa comunque fare
        a = sp.sparse.csr_matrix((0, num_points_total*interesting_dimensions))
        mis = np.array([])
        B = sp.sparse.dok_matrix(np.zeros((0, sens_count[len(sens_count)-1])))
        return mis, a, B
       
    #smembro gli elementi delle liste in input
    lista_LOS_sens_ridotta = [sens.LOS for sens in lista_sensori]
    lista_mis_sens_ridotta = [sens.mis for sens in lista_sensori]

    axes_names = cubo.attrs.get("axes_names")
    intervals = [np.array(cubo.coords[name]) for name in axes_names]
    num_sensori = len(lista_sensori)
     
    #inizializzo la matrice che comprende tutti i polinomi con le coordinate: mi serve sapere il numero di colonne totale, 
    # ovvero quanto spazio verrà occupato in totale da tutti i sensori. Il numero di colonne è stimato per eccesso
    B = sp.sparse.dok_matrix((num_sensori*valid_points, sens_count[len(sens_count)-1]))

    #coordinate dei punti validi
    block_coords = da.where(maschera>0) #relative    
    
    #matrice sparsa dove a ogni punto valido è associato un numero naturale 
    sp_mat = sparse.COO([block_coords[0], block_coords[1], block_coords[2]], np.arange(valid_points)+1, fill_value=-1)

    #inizializzazione
    misure = []
    a_list = []
    counter = 0
    
    for i in np.arange(num_sensori): #per ogni sensore
        
        #check: entro solo se il sensore ha i dati di almeno un punto valido
        #if np.isfinite(lista_mis_sens_ridotta[i].data.any()):
        if da.isfinite(lista_mis_sens_ridotta[i].all()):
        #if da.isfinite(lista_mis_sens_ridotta[i][block_coords].all()):
            
            #intersezione tra i punti validi e i punti di cui il sensore ha rilevazioni
            #coordinates = np.where((np.isfinite(lista_mis_sens_ridotta[i].values) == True) & (maschera>0))
            coordinates = np.where((da.isfinite(lista_mis_sens_ridotta[i]) == True) & (maschera>0))

            #numero di punti validi nel sensore
            n_points = len(coordinates[0])
            
            #indice relativo dei punti validi analizzati dal sensore
            list_points = sp_mat[coordinates[0], coordinates[1], coordinates[2]]-1
            list_points = list_points.todense() #3m42s
            
            #grado del polinomio che funge da filtro spaziale
            grado = lista_sensori[i].grado
            
            #coordinate assolute dei punti validi analizzati dal sensore
            absolute_coordinates = np.array([intervals[0][coordinates[0]], intervals[1][coordinates[1]], intervals[2][coordinates[2]]])

            #restretch in the relative domain
            absolute_coordinates = np.array([(absolute_coordinates[i, :]-BB[i, 0])*(BB_rel[i, 1]-BB_rel[i, 0])/(BB[i, 1]-BB[i, 0])+BB_rel[i, 0] for i in range(absolute_coordinates.shape[0])])

            #creo una striscia di A relativa al sensore: uso tante righe quante il numero di punti validi nel sensore e per ogni riga 
            #occupo 3 colonne. l'i_esimo punto valido relativo a questo sensore occuperà la riga i alle colonne (numero di punti nei 
            # chunk precedenti a questo + indice relativo del sensore), e il o i successivi in base a quante sono le componenti 
            #della LOS a cui siamo interessati, che saranno i dati che andremo a piazzare
            rows_a = da.arange(n_points)[:, None]*da.ones(interesting_dimensions)[None, :]
            rows_a = rows_a.reshape((n_points*interesting_dimensions, ))

            columns_a = int(conto_punti[num_chunk])*interesting_dimensions+list_points[:, None]*interesting_dimensions+ np.arange(interesting_dimensions)[None, :]
            columns_a = columns_a.reshape((n_points*interesting_dimensions, ))
            
            data_a = lista_LOS_sens_ridotta[i].values[coordinates[0], coordinates[1], coordinates[2], :]
            data_a = data_a.reshape(n_points*interesting_dimensions)

            a = sp.sparse.coo_array((data_a, (da.from_array([rows_a, columns_a]))), shape = (n_points, interesting_dimensions*num_points_total))
            a_list.append(a)
            
            #inserisco i valori in B
            B[counter:counter+n_points, sens_count[i]:sens_count[i+1]] = sp.sparse.coo_array(polynomial(np.transpose(absolute_coordinates), grado))

            #il termine noto comprende le misurazioni
            misure.append(list(lista_mis_sens_ridotta[i].values[coordinates]))
                        
            #il counter serve per sapere quali righe di B sono effettivamente usate
            counter += n_points

    #assemblo A e il termine noto
    mis = np.hstack(misure).flatten() 
    #A = sp.sparse.vstack([a for a in a_list]) #3m42s #5m26s
    A=sp.sparse.vstack(a_list) #5m32s #3m12 (?!)
    #taglio B
    B=B[0:counter, :]
    return mis, A, B

In [31]:
def get_SVD(lista, soglia=0.002): #TBC
    '''
    Questo modulo ritorna un vettore che indica il numero di dimensioni indipendenti che è possibie ricostruire 
    combinando i dati di tutte le possibili combinazioni di vettori. 
    A ogni sensore viene associato un bit: se ad esempio abbiamo 4 sensori avremo tutte le possibili combinazioni di 
    0 e 1 con 4 cifre, ovvero in binario tutti i numeri da 1 a 15. Ad esempio,in posizione 7 - in binario 0111 - 
    (partendo da 0 ovviamente) avremo il numero di dimensioni che è possibile ricreare combinando i dati ricevuti 
    dai sensori 1, 2 e 3 (sempre partendo da 0), nell'ordine in cui sono passati nella lista -la lista delle LOS medie.
    Il numero di combinazioni è dato dal numero di valori singolari della matrice delle LOS che supera la soglia 
    ricevuta in input.
    '''
    
    #salvo la lista
    LOS = da.array(lista)
    LOS_ndims = LOS.shape[1]
    
    #normalizzo
    L = da.sqrt(da.sum(da.square(LOS), axis = 1))
    LOS = LOS/(L[:, None]*da.ones(LOS_ndims)[None, :])
    
    #creo le combinazioni di sensori
    combinations = np.array(list(itertools.product(range(2), repeat=len(lista))))
    combinations = combinations[:, ::-1]
    num_comb = 2**len(lista)
    
    #inizializzo
    SVD = np.zeros(num_comb)
    
    #calcolo i valori singolari
    for i in range(num_comb):
        n = sp.linalg.svdvals(np.multiply(LOS, combinations[i][:, None]*np.ones(LOS_ndims)[None, :]).compute())
        SVD[i] = np.count_nonzero(n > soglia) 

    return SVD

In [32]:
def get_mask(lista_data, interesting_dimensions):
    '''
    il modulo prende in input la lista dei dataset che riassumono i dati raccolti da ogni sensore e 
    restituisce una maschera dove a ogni punto si accocia il numero 1 se il numero di dimensioni
    che è possibile ricostruire è minore o uguale al numero di dimensioni richiesto passato in 
    input, 0 altrimenti.
    '''
    
    #inizializzo
    lista_LOS_medie = []
    lista_valori = []
    sensors_mask = 0
    
    #creo la lista delle LOS medie e dei termini noti
    for el in lista_data:
        lista_LOS_medie.append(el.attrs.get('LOS_media'))
        lista_valori.append(el.mis)

    if len(lista_LOS_medie[0]) < interesting_dimensions:
        print("Informazioni insufficienti: le dimensioni della LOS non sono sufficienti a ricostruire le dimensioni richieste")

    SVD = get_SVD(lista_LOS_medie)

    #creo una maschera che indica i sensori
    for i, ri in enumerate(lista_valori):
        sensors_mask += np.isfinite(ri) * 2**i

    #creo la maschera che riporta le dimensioni ricostruibili nel punto
    rank_mask = SVD[sensors_mask]

    return da.from_array(rank_mask//interesting_dimensions)

In [43]:
def simulo_da_soluzione(n_punti, dim, grado, n_sensori, cartella):

    LOS_medie = 20*da.random.random_sample((n_sensori, dim))

    #normalizzo
    L = da.sqrt(da.sum(da.square(LOS_medie), axis = 1))
    LOS_medie = LOS_medie/(L[:, None]*da.ones(dim)[None, :])
               
    coeff = []
    #scelgo i coefficienti
    for i in range(n_sensori):
        coeff.append(np.random.rand(len(list(set(product(set(range(grado + 1)),repeat = 3))))))
        
    nomi_file=[]
    for i in range(n_sensori):
        nomi_file.append(cartella+"/"+str(i))
        
    for i in range(n_sensori):
        simulo_sensore_da_soluzione(n_punti, nomi_file[i], LOS_medie[i, :], grado)
        
    cube = reticolo(nomi_file, np.array([1, 2, 3]))
    
    lista_sens = []
    
    for i in range(n_sensori):
        lista_sens.append(clusterizzazione(nomi_file[i], cube, grado))
        
    r = get_mask(lista_sens, dim)
    
    #conto i coefficienti utili per ogni sensore
    sens_count = sens_counter(lista_sens)#lo faccio qui o in ogni chunk? indifferente

    #intervallo di divisione dei chunk
    chunk_interval = np.array([10, 10, 10])
    #chunk_interval = np.array([5, 5, 5])

    #conto i punti in ogni chunk
    points_count = points_counter(chunk_interval, r)
    
    axes_names = cube.attrs.get('axes_names')  
    ch = {axes_names[i]: chunk_interval[i] for i in range(len(axes_names))}
    axes = [cube.coords[name] for name in axes_names]
    masch_ds = xr.DataArray(name='masch_ds', data=r,
                    dims=axes_names, coords={axes_names[i]: axes[i] for i in range(len(axes_names))})

    cube = cube.assign_attrs({"interesting_dimensions" : dim, "conto_punti" : points_count})
    
    obj = [cube, masch_ds]

    for el in lista_sens:
        obj.append(el)

    mis_new, A, B = map_blocks(chunk_matrices_tagliata, obj, chunksizes = ch)

    A_comp = sp.sparse.vstack(A)
    B_comp = sp.sparse.vstack(B)

    Mat_new = sp.sparse.hstack([A_comp, B_comp])
    Mat_new = sp.sparse.vstack([Mat_new, sp.sparse.eye(Mat_new.shape[1])*10**(-4)])
    
    b = Mat_new@soluzione
    
    #rumore a b
    
    

In [52]:
simulo_da_soluzione(4, 3, 1, 2, "p")

[1249.93829217  539.58431843  345.92117056  449.4605099 ]
[347.28816484 465.70081698 411.12571739 622.22011723]


KeyError: 'misure'

In [ ]:
n_punti = 1000000 #3m32s (11m) 1m 2mln 36m5s 100k 1m27/3m
#N.B. provo con 100k punti perché ci sono più chunk vuoti e vedo se va

#scelgo le LOS medie
LOS_mean_1 = da.array([0.1, 1, 1])
LOS_mean_2 = da.array([1, 2, 2])
LOS_mean_3 = da.array([0, 3, 5])
LOS_mean_4 = da.array([3, 1, 4])

#scelgo i nomi degli assi
axes_names = ["vert", 'est', "nord"]

num_sensori = 4

In [ ]:
#simulo i sensori con le date istruzioni
simulo_sensore(n_punti, 'tmp/sens_1', LOS_mean_1, axes_names)
simulo_sensore(n_punti, 'tmp/sens_2', LOS_mean_2, axes_names)
simulo_sensore(n_punti, 'tmp/sens_3', LOS_mean_3, axes_names)
simulo_sensore(n_punti, 'tmp/sens_4', LOS_mean_4, axes_names)

In [ ]:
#creo in reticolo date le coordinate dei punti contenuti nei csv
cube = reticolo(['tmp/sens_1', 'tmp/sens_2', 'tmp/sens_3', 'tmp/sens_4'], np.array([1, 2, 3]), axes_names = axes_names)

In [ ]:
#creo i dataset #http://127.0.0.1:8787/
data_set_1 =clusterizzazione('tmp/sens_1', cube, 1)
data_set_2 =clusterizzazione('tmp/sens_2', cube, 1)
data_set_3 =clusterizzazione('tmp/sens_3', cube, 2)
data_set_4 =clusterizzazione('tmp/sens_4', cube, 2)

In [ ]:
#creo la maschera di sensori e la maschera con rank delle matrici delle LOS 
# medie dei sensori che raccolgono dati in ogni punto
lista_sens = [data_set_1, data_set_2, data_set_3, data_set_4]
l_s = data_set_1, data_set_2, data_set_3, data_set_4

#dimensioni che consideriamo delle LOS
interesting_dimensions = 3
r = get_mask(lista_sens, interesting_dimensions)

In [ ]:
#conto i coefficienti utili per ogni sensore
sens_count = sens_counter(lista_sens)#lo faccio qui o in ogni chunk? indifferente

#intervallo di divisione dei chunk
chunk_interval = np.array([10, 10, 10])
#chunk_interval = np.array([5, 5, 5])

#conto i punti in ogni chunk
points_count = points_counter(chunk_interval, r)

In [ ]:
#calcolo le porzioni di matrice e termine noto riferite a un determinato chunk
num_chunk = 75
#termine_noto, A, B = chunk_matrices(num_chunk, chunk_interval, lista_sens, r, cube, interesting_dimensions, points_count)

In [ ]:
ch = {axes_names[i]: chunk_interval[i] for i in range(len(axes_names))}
axes = [cube.coords[name] for name in axes_names]
masch_ds = xr.DataArray(name='masch_ds', data=r,
                    dims=axes_names, coords={axes_names[i]: axes[i] for i in range(len(axes_names))})

cube = cube.assign_attrs({"interesting_dimensions" : interesting_dimensions, "conto_punti" : points_count})

In [ ]:
obj = [cube, masch_ds]

for el in l_s:
    obj.append(el)

mis_new, A, B = map_blocks(chunk_matrices_tagliata, obj, chunksizes = ch)

A_comp = sp.sparse.vstack(A)
B_comp = sp.sparse.vstack(B)

Mat_new = sp.sparse.hstack([A_comp, B_comp])
Mat_new = sp.sparse.vstack([Mat_new, sp.sparse.eye(Mat_new.shape[1])*10**(-4)])

In [ ]:
M = np.concatenate(mis_new)
M = np.concatenate([M, np.zeros(Mat_new.shape[1])])
print(Mat_new.shape)
y = sp.sparse.linalg.lsqr(Mat_new, M)
z = sp.sparse.linalg.lsqr(Mat_new.transpose()@Mat_new, Mat_new.transpose()@M)
print(y)
U = sp.sparse.csr_array(Mat_new.transpose()@Mat_new)
print(U.shape)
#print(np.linalg.matrix_rank(U))
print(sp.sparse.csgraph.structural_rank(U))
#print(sp.linalg.expm_cond(U))

(1219303, 524431)
(array([ 5.49385713e+00, -3.08623157e+00,  2.94829332e+00, ...,
        1.81279359e-03,  1.34905804e-03, -9.54884672e-04]), 2, 145, 1141.4174263247135, 1141.4174263247135, 45106.743785995706, 270349.2164269757, 38.74163188011378, 5099.7126040530175, array([0., 0., 0., ..., 0., 0., 0.]))
(524431, 524431)
524431


In [ ]:
solver = pypardiso.PyPardisoSolver()
print(Mat_new.shape)
print(M.shape)
x = solver.solve(Mat_new.transpose()@Mat_new, Mat_new.transpose()@M)

(1219303, 524431)
(1219303,)


In [ ]:
print(x)
print(max(y[0]-x))
print(max(abs(z[0]-x)))
print(min(x))
print(max(y[0]))

[ 4.33562100e+00 -4.92426334e+00  2.31429862e+00 ...  7.85295757e-03
 -8.29171662e-04 -1.50826414e-02]
8482.281305203422
8538.168963520813
-8485.464443226072
47.03046372175178


Nel primo test scelgo 10 punti e provo a simulare i dati: i primi 3 sensori si riferiscono a 
tutti e 10 i punti e le LOS_medie hanno rank 3. Il 4 sensore è la copia del 3 con un punto in meno:
serve per vedere se all'interno del chunk col punto mancante le colonne vengono assegnate bene.
Inoltre, nel punto mancante le LOS medie degli altri sensori hanno sempre rank 3:
serve per capire se i rank sono associati in maniera giusta (controllare la creazione della 
maschera dei sensori e del vettore con i valori singolari)

In [ ]:
#10 punti
#simulo_sensore a mano
#punti_1 = np.array([[1.2, 0.9, 0.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.4, 5.1, 3.9],
#                    [6.1, 6.2, 4.9], [7.1, 6.2, 4.9], [4.8, 8.1, 1.7], 
#                    [8.2, 4.9, 0.8], [9, 0.8, 2.3], [7.9, 5.8, 2.3]])
#punti_2 = np.array([[1.3, 0.7, 0.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.1, 5.1, 3.6],
#                    [6.3, 6.3, 4.9], [7.1, 6.2, 4.9], [4.8, 8.4, 1.9], 
#                    [8.4, 4.7, 0.8], [9, 0.8, 2.3], [7.9, 5.9, 2.3]])
#punti_3 = np.array([[1.3, 0.7, 0.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.1, 5.1, 3.6],
#                    [6.3, 6.3, 4.9], [7.1, 6.2, 4.9], [4.8, 8.4, 1.9], 
#                    [8.4, 4.7, 0.8], [9, 0.8, 2.3], [7.9, 5.9, 2.3]])
#punti_4 = np.array([[1.3, 0.7, 0.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.1, 5.1, 3.6],
#                    [6.3, 6.3, 4.9], [4.8, 8.4, 1.9], 
#                    [8.4, 4.7, 0.8], [9, 0.8, 2.3], [7.9, 5.9, 2.3]])

In questo secondo set di coordinate il dominio viene diviso in 8 chunks (2 lungo ogni
dimensione) e serve vedere se la divisione in chunks e come vengono numerati combacia
con quanto descritto in chunk_matrices. Inoltre, sia qui che nell'esempio precedente, 
ci sono dei chunk vuoti

In [ ]:
punti_1 = np.array([[1.2, 0.9, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.4, 5.1, 3.9],
                    [6.1, 6.2, 6.9], [7.1, 6.2, 4.9], [4.8, 8.1, 1.7], 
                    [8.2, 4.9, 0.8], [9, 0.8, 2.3], [7.9, 5.8, 8.3]])
punti_2 = np.array([[1.3, 0.7, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.1, 5.1, 3.6],
                    [6.3, 6.3, 6.9], [7.1, 6.2, 4.9], [4.8, 8.4, 1.9], 
                    [8.4, 4.7, 0.8], [9, 0.8, 2.3], [7.9, 5.9, 8.3]])
punti_3 = np.array([[1.3, 0.7, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.1, 5.1, 3.6],
                    [6.3, 6.3, 6.9], [7.1, 6.2, 4.9], [4.8, 8.4, 1.9], 
                    [8.4, 4.7, 0.8], [9, 0.8, 2.3], [7.9, 5.9, 8.3]])
punti_4 = np.array([[1.3, 0.7, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.1, 5.1, 3.6],
                    [6.3, 6.3, 6.9], [4.8, 8.4, 1.9], 
                    [8.4, 4.7, 0.8], [9, 0.8, 2.3], [7.9, 5.9, 8.3]])
LOS_1 = np.array([[0.75, 0.13, 0.65], [0.75, 0.13, 0.65], [0.74, 0.14, 0.64], 
                  [0.74, 0.14, 0.64], [0.74, 0.14, 0.64], [0.76, 0.14, 0.66], 
                  [0.76, 0.14, 0.66], [0.75, 0.12, 0.66], [0.75, 0.12, 0.66],
                  [0.75, 0.12, 0.66]])
LOS_2 = np.array([[0.85, 0.08, 0.52], [0.86, 0.1, 0.52], [0.85, 0.1, 0.52],
                  [0.85, 0.1, 0.52], [0.85, 0.1, 0.52], [0.86, 0.11, 0.52],
                  [0.85, 0.1, 0.52], [0.85, 0.1, 0.52], [0.85, 0.1, 0.54],
                  [0.85, 0.09, 0.54]])
LOS_3 = np.array([[1.85, 0.08, 0.92], [1.86, 0.1, 0.92], [1.85, 0.1, 0.92],
                  [1.85, 0.1, 0.92], [1.85, 0.1, 0.92], [1.86, 0.11, 0.92],
                  [1.85, 0.1, 0.92], [1.85, 0.1, 0.92], [1.85, 0.1, 0.94],
                  [1.85, 0.09, 0.94]])
LOS_4 = np.array([[1.85, 0.08, 0.92], [1.86, 0.1, 0.92], [1.85, 0.1, 0.92],
                  [1.85, 0.1, 0.92], [1.85, 0.1, 0.92], 
                  [1.85, 0.1, 0.92], [1.85, 0.1, 0.92], [1.85, 0.1, 0.94],
                  [1.85, 0.09, 0.94]])

dim = LOS_4.shape[1]
#normalizzo
L_1 = np.sqrt(np.sum(np.square(LOS_1), axis = 1))
L_2 = np.sqrt(np.sum(np.square(LOS_2), axis = 1))
L_3 = np.sqrt(np.sum(np.square(LOS_3), axis = 1))
L_4 = np.sqrt(np.sum(np.square(LOS_4), axis = 1))
LOS_1 = LOS_1/(L_1[:, None]*np.ones(dim)[None, :])
LOS_2 = LOS_2/(L_2[:, None]*np.ones(dim)[None, :])
LOS_3 = LOS_3/(L_3[:, None]*np.ones(dim)[None, :])
LOS_4 = LOS_4/(L_4[:, None]*np.ones(dim)[None, :])

d= np.array([[0.5, 0.5, 0.5], [1, 1, 1], [0.3, 0.3, 0.3], [0.4, 0.2, 0.1], [0.3, 0.7, 9.1],
             [0.5, 0.5, 0.5], [1, 1, 1], [0.3, 0.3, 0.3], [0.4, 0.2, 0.1], [0.3, 0.7, 9.1]])
d_4 = np.array([[0.5, 0.5, 0.5], [1, 1, 1], [0.3, 0.3, 0.3], [0.4, 0.2, 0.1], [0.3, 0.7, 9.1],
             [1, 1, 1], [0.3, 0.3, 0.3], [0.4, 0.2, 0.1], [0.3, 0.7, 9.1]])

#coefficienti per il filtro polinomiale spaziale
coeff = np.array([1, 1, 2, 2, 3, 3, 4, 4, 5, 5])
coeff_4 = np.array([1, 1, 2, 2, 3, 4, 4, 5, 5])
d_proiettata_1 = np.zeros((LOS_1.shape[0]))
d_proiettata_2 = np.zeros((LOS_1.shape[0]))
d_proiettata_3 = np.zeros((LOS_1.shape[0]))
d_proiettata_4 = np.zeros((LOS_4.shape[0]))

for i in range(LOS_1.shape[0]):
    d_proiettata_1[i] = np.dot(LOS_1[i, :], d[i, :])
    d_proiettata_2[i] = np.dot(LOS_2[i, :], d[i, :])
    d_proiettata_3[i] = np.dot(LOS_3[i, :], d[i, :])
    if i<9:
        d_proiettata_4[i] = np.dot(LOS_4[i, :], d_4[i, :])
    
df1 = pd.DataFrame({'x': punti_1[:, 0], 'y': punti_1[:, 1], 'z': punti_1[:, 2], 
                'LOS_x': LOS_1[:, 0], 'LOS_y': LOS_1[:, 1], 'LOS_z': LOS_1[:, 2], 'misure': d_proiettata_1+coeff})

df2 = pd.DataFrame({'x': punti_2[:, 0], 'y': punti_2[:, 1], 'z': punti_2[:, 2], 
                'LOS_x': LOS_2[:, 0], 'LOS_y': LOS_2[:, 1], 'LOS_z': LOS_2[:, 2], 'misure': d_proiettata_2+coeff})

df3 = pd.DataFrame({'x': punti_3[:, 0], 'y': punti_3[:, 1], 'z': punti_3[:, 2], 
                'LOS_x': LOS_3[:, 0], 'LOS_y': LOS_3[:, 1], 'LOS_z': LOS_3[:, 2], 'misure': d_proiettata_3+coeff})

df4 = pd.DataFrame({'x': punti_4[:, 0], 'y': punti_4[:, 1], 'z': punti_4[:, 2], 
                'LOS_x': LOS_4[:, 0], 'LOS_y': LOS_4[:, 1], 'LOS_z': LOS_4[:, 2], 'misure': d_proiettata_4+coeff_4})

ddf1 = dask.dataframe.from_pandas(df1).repartition(partition_size='10MB')
ddf2 = dask.dataframe.from_pandas(df2).repartition(partition_size='10MB')
ddf3 = dask.dataframe.from_pandas(df3).repartition(partition_size='10MB')
ddf4 = dask.dataframe.from_pandas(df4).repartition(partition_size='10MB')

if os.path.isdir('tmp/prova_1'):
    for file in os.listdir('tmp/prova_1'):
        os.remove('tmp/prova_1'+'/'+file)
    os.removedirs('tmp/prova_1')

if os.path.isdir('tmp/prova_2'):
    for file in os.listdir('tmp/prova_2'):
        os.remove('tmp/prova_2'+'/'+file)
    os.removedirs('tmp/prova_2')
    
if os.path.isdir('tmp/prova_3'):
    for file in os.listdir('tmp/prova_3'):
        os.remove('tmp/prova_3'+'/'+file)
    os.removedirs('tmp/prova_3')
    
if os.path.isdir('tmp/prova_4'):
    for file in os.listdir('tmp/prova_4'):
        os.remove('tmp/prova_4'+'/'+file)
    os.removedirs('tmp/prova_4')
    
ddf1.to_csv('tmp/prova_1', mode='x')
ddf2.to_csv('tmp/prova_2', mode='x')
ddf3.to_csv('tmp/prova_3', mode='x')
ddf4.to_csv('tmp/prova_4', mode='x')

['/home/debora/rose-l-ard/poi/tmp/prova_4/0.part']

In [ ]:
cube1 = reticolo(['tmp/prova_1', 'tmp/prova_2', 'tmp/prova_3', 'tmp/prova_4'], np.array([1, 1, 1]))

data_set_prova_1=clusterizzazione('tmp/prova_1', cube1, 1)
data_set_prova_2=clusterizzazione('tmp/prova_2', cube1, 1)
data_set_prova_3=clusterizzazione('tmp/prova_3', cube1, 1)
data_set_prova_4=clusterizzazione('tmp/prova_4', cube1, 1)

lista_sens_prova = [data_set_prova_1, data_set_prova_2, data_set_prova_3, data_set_prova_4]
interesting_dimensions = 3

r_prova = get_mask(lista_sens_prova, interesting_dimensions) 

In [ ]:
dim = 3
sens_count = sens_counter(lista_sens_prova)
chunk_interval = np.array([5, 5, 5])

points_count_prova = points_counter(chunk_interval, r_prova)

In [ ]:
axes_names = cube1.axes_names
l_s_prova = data_set_prova_1, data_set_prova_2, data_set_prova_3, data_set_prova_4
ch_prova = {axes_names[i]: chunk_interval[i] for i in range(len(axes_names))}
axes = [cube1.coords[name] for name in axes_names]
masch_ds_prova = xr.DataArray(name='masch_ds', data=r_prova,
                    dims=axes_names, coords={axes_names[i]: axes[i] for i in range(len(axes_names))})

cube1 = cube1.assign_attrs({"interesting_dimensions" : interesting_dimensions, "conto_punti" : points_count_prova})

obj_prova = [cube1, masch_ds_prova]

for el in l_s_prova:
    obj_prova.append(el)

In [ ]:
mis_new, A, B = map_blocks(chunk_matrices_tagliata, obj_prova, chunksizes = ch_prova)

A_comp = sp.sparse.vstack(A)
B_comp = sp.sparse.vstack(B)
Mat_new = sp.sparse.hstack([A_comp, B_comp])
M = np.concatenate(mis_new)

2025-03-04 12:25:29 [warning  ] No global client found a new one will be created
2025-03-04 12:25:29 [debug    ] <Cluster info: type=LOCAL, n_jobs=22, job_memory=1444500638>
2025-03-04 12:25:31 [debug    ] Selected scatter strategy: once-and-for-all
2025-03-04 12:25:31 [debug    ] Number of blocks to be processed: 8
no valid points in chunk 3
no valid points in chunk 5


/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/pytho

[[5.]
 [8.]
 [2.]]
[[ 0.        ]
 [ 5.        ]
 [-3.57142857]]
[[6. 8.]
 [6. 6.]
 [7. 8.]]
[[1.25       3.75      ]
 [2.14285714 2.14285714]
 [3.57142857 5.        ]]
[[6. 8.]
 [6. 6.]
 [7. 8.]]
[[7.]
 [6.]
 [5.]]


/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/pytho

[[5.]
 [8.]
 [2.]]
[[1. 2. 3.]
 [2. 4. 5.]
 [3. 2. 4.]]
[[1.25       3.75      ]
 [2.14285714 2.14285714]
 [3.57142857 5.        ]]
[[2.5       ]
 [2.14285714]
 [0.71428571]]
[[ 0.        ]
 [ 5.        ]
 [-3.57142857]]
[[6. 8.]
 [6. 6.]
 [7. 8.]]
[[7.]
 [6.]
 [5.]]
[[5.]
 [8.]
 [2.]]
[[1.25       3.75      ]
 [2.14285714 2.14285714]
 [3.57142857 5.        ]]
[[2.5       ]
 [2.14285714]
 [0.71428571]]
[[ 0.        ]
 [ 5.        ]
 [-3.57142857]]
[[6. 8.]
 [6. 6.]
 [7. 8.]]
[[-5.         -3.75       -2.5       ]
 [-3.57142857 -0.71428571  0.71428571]
 [-2.14285714 -3.57142857 -0.71428571]]
[[7.]
 [6.]
 [5.]]
[[5.]
 [8.]
 [2.]]
[[1.25       3.75      ]
 [2.14285714 2.14285714]
 [3.57142857 5.        ]]
[[2.5       ]
 [2.14285714]
 [0.71428571]]
[[ 0.        ]
 [ 5.        ]
 [-3.57142857]]
[[1. 2. 3.]
 [2. 4. 5.]
 [3. 2. 4.]]


/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/pytho

[]
[[-5.         -3.75       -2.5       ]
 [-3.57142857 -0.71428571  0.71428571]
 [-2.14285714 -3.57142857 -0.71428571]]
[]
[[1. 2. 3.]
 [2. 4. 5.]
 [3. 2. 4.]]
[[-5.         -3.75       -2.5       ]
 [-3.57142857 -0.71428571  0.71428571]
 [-2.14285714 -3.57142857 -0.71428571]]
[[1. 2. 3.]
 [2. 4. 5.]
 [3. 2. 4.]]
[[-5.         -3.75       -2.5       ]
 [-3.57142857 -0.71428571  0.71428571]
 [-2.14285714 -3.57142857 -0.71428571]]


/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/pytho

[[8. 9.]
 [5. 1.]
 [1. 2.]]
[[1.]
 [1.]
 [6.]]
[[-5.        ]
 [-5.        ]
 [ 2.14285714]]
[[1.]
 [1.]
 [6.]]
[[-5.        ]
 [-5.        ]
 [ 2.14285714]]
[[ 3.75        5.        ]
 [ 0.71428571 -5.        ]
 [-5.         -3.57142857]]
[[8. 9.]
 [5. 1.]
 [1. 2.]]
[[1.]
 [1.]
 [6.]]
[[-5.        ]
 [-5.        ]
 [ 2.14285714]]
[[ 3.75        5.        ]
 [ 0.71428571 -5.        ]
 [-5.         -3.57142857]]
[[8. 9.]
 [5. 1.]
 [1. 2.]]


/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/pytho

[[1.]
 [1.]
 [6.]]
[[ 3.75        5.        ]
 [ 0.71428571 -5.        ]
 [-5.         -3.57142857]]
[[-5.        ]
 [-5.        ]
 [ 2.14285714]]
[[8. 9.]
 [5. 1.]
 [1. 2.]]
[[ 3.75        5.        ]
 [ 0.71428571 -5.        ]
 [-5.         -3.57142857]]


In [ ]:
y = sp.sparse.linalg.lsqr(Mat_new, M)
z = sp.sparse.linalg.lsqr(Mat_new.transpose()@Mat_new, Mat_new.transpose()@M)
solver = pypardiso.PyPardisoSolver()
print(Mat_new.shape)
print(M.shape)
x = solver.solve(Mat_new.transpose()@Mat_new, Mat_new.transpose()@M)
print(x)
print(max(y[0]-x))
print(max(abs(z[0]-x)))
print(min(x))

(39, 46)
(39,)
[ 3.89683560e+00  2.02365095e+01 -4.99156971e+00  5.47942002e+00
  2.98909812e+01 -9.38327208e+00  1.99801425e+00  9.68829636e-01
  1.04453290e+00 -8.55196932e-01 -1.98612925e+01  7.48314638e+00
  5.13287045e+00  1.38145266e+01 -4.26341645e-01  4.76151428e+00
  2.12514883e+01 -2.76832820e+00  6.41324926e+00  2.25392228e+01
 -3.56894201e+00  1.38158251e+00 -1.26712664e+01  6.90614404e+00
 -2.74571643e+00 -4.26428503e+01  2.68672748e+01 -2.36478108e+00
 -6.62639866e+01  3.21331032e+01  6.84059945e-02 -2.94595800e-02
 -2.06642809e-02 -3.31031411e-03  1.81896346e-01  1.13449709e-01
  3.52899920e-02  1.98532051e-02  7.74882797e-02 -5.25774802e-02
  1.06797730e-02  3.64166602e-03  7.74882797e-02 -5.25774802e-02
  1.06797730e-02  3.64166602e-03]
67.90731274718891
66.78594132804137
-66.26398661886365


In [ ]:
a = np.array([1, 1, 1])
n = np.sqrt(3)
print(1/n)


0.5773502691896258


In [ ]:
termine_noto1, A1, B1 = chunk_matrices(0, chunk_interval, lista_sens_prova, r_prova, cube1, interesting_dimensions, points_count_prova)
termine_noto2, A2, B2 = chunk_matrices(1, chunk_interval, lista_sens_prova, r_prova, cube1, interesting_dimensions, points_count_prova)
termine_noto3, A3, B3 = chunk_matrices(2, chunk_interval, lista_sens_prova, r_prova, cube1, interesting_dimensions, points_count_prova)
termine_noto4, A4, B4 = chunk_matrices(3, chunk_interval, lista_sens_prova, r_prova, cube1, interesting_dimensions, points_count_prova)
termine_noto5, A5, B5 = chunk_matrices(4, chunk_interval, lista_sens_prova, r_prova, cube1, interesting_dimensions, points_count_prova)
termine_noto6, A6, B6 = chunk_matrices(5, chunk_interval, lista_sens_prova, r_prova, cube1, interesting_dimensions, points_count_prova)
termine_noto7, A7, B7 = chunk_matrices(6, chunk_interval, lista_sens_prova, r_prova, cube1, interesting_dimensions, points_count_prova)
termine_noto8, A8, B8 = chunk_matrices(7, chunk_interval, lista_sens_prova, r_prova, cube1, interesting_dimensions, points_count_prova)

#compongo le 
A = sp.sparse.vstack([A1, A2, A3, A4, A5, A6, A7, A8])
B = sp.sparse.vstack([B1, B2, B3, B4, B5, B6, B7, B8])
Mat = sp.sparse.hstack([A, B])
termine_noto = np.hstack([termine_noto1, termine_noto2, termine_noto3, termine_noto4, termine_noto5, termine_noto6, termine_noto7, termine_noto8])

TypeError: 'NoneType' object is not subscriptable

ora consideriamo solo due dimensioni delle LOS, simulando i punti con coordinate random con 4 sensori.

In [ ]:
n_punti = 100000
dim=2

LOS_mean_1 = da.array([0.1, 1])
LOS_mean_2 = da.array([1, 2])
LOS_mean_3 = da.array([0, 3])
LOS_mean_4 = da.array([3, 1])
axes_names = ["vert", 'est', "nord"]
simulo_sensore(n_punti, 'tmp/prova2_1', LOS_mean_1, axes_names)
simulo_sensore(n_punti, 'tmp/prova2_2', LOS_mean_2, axes_names)
simulo_sensore(n_punti, 'tmp/prova2_3', LOS_mean_3, axes_names)
simulo_sensore(n_punti, 'tmp/prova2_4', LOS_mean_4, axes_names)

num_sensori = 4

In [ ]:
cube2 = reticolo(['tmp/prova2_1', 'tmp/prova2_2', 'tmp/prova2_3', 'tmp/prova2_4'], np.array([1, 2, 3]), axes_names = axes_names, ndims = 2)
BB2= cube2.attrs.get("BB")

In [ ]:
data_set_prova2_1=clusterizzazione('tmp/prova2_1', cube2, 1)
data_set_prova2_2=clusterizzazione('tmp/prova2_2', cube2, 1)
data_set_prova2_3=clusterizzazione('tmp/prova2_3', cube2, 1)
data_set_prova2_4=clusterizzazione('tmp/prova2_4', cube2, 1)
lista_sens_prova2 = [data_set_prova2_1, data_set_prova2_2, data_set_prova2_3, data_set_prova2_4]

interesting_dimensions_prova2 = 2
r_prova2 = get_mask(lista_sens_prova2, interesting_dimensions_prova2)

In [ ]:
sens_count_prova2 = sens_counter(lista_sens_prova2)
chunk_interval_prova2 = np.array([10, 10, 10])

points_count_prova2 = points_counter(chunk_interval_prova2, r_prova2)

In [ ]:
num_chunk = 75
termine_noto_prov2, A_prova2, B_prova2 = chunk_matrices(num_chunk, chunk_interval_prova2, lista_sens_prova2, r_prova2, cube2, interesting_dimensions_prova2, points_count_prova2)

simulo manualmente 10 punti utilizzando solo 2 sensori e 2 coordinate delle LOS

In [ ]:
#10 punti
#simulo_sensore a mano

punti_1 = np.array([[1.2, 0.9, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.4, 5.1, 3.9],
                    [6.1, 6.2, 6.9], [7.1, 6.2, 4.9], [4.8, 8.1, 1.7], 
                    [8.2, 4.9, 0.8], [9, 0.8, 2.3], [7.9, 5.8, 8.3]])
punti_2 = np.array([[1.3, 0.7, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.1, 5.1, 3.6],
                    [6.3, 6.3, 6.9], [7.1, 6.2, 4.9], [4.8, 8.4, 1.9], 
                    [8.4, 4.7, 0.8], [9, 0.8, 2.3], [7.9, 5.9, 8.3]])
LOS_1 = np.array([[0.75, 0.13], [0.75, 0.13], [0.74, 0.14], 
                  [0.74, 0.14], [0.74, 0.14], [0.76, 0.14], 
                  [0.76, 0.14], [0.75, 0.12], [0.75, 0.12],
                  [0.75, 0.12]])
LOS_2 = np.array([[0.85, 0.08], [0.86, 0.1], [0.85, 0.1],
                  [0.85, 0.1], [0.85, 0.1], [0.86, 0.11],
                  [0.85, 0.1], [0.85, 0.1], [0.85, 0.1],
                  [0.85, 0.0]])

dim = 2

#normalizzo
L_1 = da.sqrt(da.sum(da.square(LOS_1), axis = 1))
L_2 = da.sqrt(da.sum(da.square(LOS_2), axis = 1))

LOS_1 = LOS_1/(L_1[:, None]*da.ones(dim)[None, :])
LOS_2 = LOS_2/(L_2[:, None]*da.ones(dim)[None, :])

d= np.array([[0.5, 0.5], [1, 1], [0.3, 0.3], [0.4, 0.2], [0.3, 0.7],
             [0.5, 0.5], [1, 1], [0.3, 0.3], [0.4, 0.2], [0.3, 0.7]])

coeff = np.array([1, 1, 2, 2, 3, 3, 4, 4, 5, 5])

d_proiettata_1 = np.zeros((LOS_1.shape[0]))
d_proiettata_2 = np.zeros((LOS_1.shape[0]))

for i in range(LOS_1.shape[0]):
    d_proiettata_1[i] = np.dot(LOS_1[i, :], d[i, :])
    d_proiettata_2[i] = np.dot(LOS_2[i, :], d[i, :])

    
#misure = da.random.random_sample(n_punti)
df1 = pd.DataFrame({'x': punti_1[:, 0], 'y': punti_1[:, 1], 'z': punti_1[:, 2], 
                'LOS_x': LOS_1[:, 0], 'LOS_y': LOS_1[:, 1], 'misure': d_proiettata_1+coeff})

df2 = pd.DataFrame({'x': punti_2[:, 0], 'y': punti_2[:, 1], 'z': punti_2[:, 2], 
                'LOS_x': LOS_2[:, 0], 'LOS_y': LOS_2[:, 1], 'misure': d_proiettata_2+coeff})

ddf1 = dd.from_pandas(df1).repartition(partition_size='10MB')
ddf2 = dd.from_pandas(df2).repartition(partition_size='10MB')

if os.path.isdir('tmp/prova3_1'):
    for file in os.listdir('tmp/prova3_1'):
        os.remove('tmp/prova3_1'+'/'+file)
    os.removedirs('tmp/prova3_1')

if os.path.isdir('tmp/prova3_2'):
    for file in os.listdir('tmp/prova3_2'):
        os.remove('tmp/prova3_2'+'/'+file)
    os.removedirs('tmp/prova3_2')

ddf1.to_csv('tmp/prova3_1', mode='x')
ddf2.to_csv('tmp/prova3_2', mode='x')

['/home/debora/rose-l-ard/poi/tmp/prova3_2/0.part']

In [ ]:
cube3 = reticolo(['tmp/prova3_1', 'tmp/prova3_2'], np.array([1, 1, 1]))

data_set_prova3_1=clusterizzazione('tmp/prova3_1', cube3, 1)
data_set_prova3_2=clusterizzazione('tmp/prova3_2', cube3, 1)

lista_sens_prova3 = [data_set_prova3_1, data_set_prova3_2]

interesting_dimensions3 = 2
r_prova3 = get_mask(lista_sens_prova3, interesting_dimensions3) 

[0.5 1.5 2.5 3.5 4.5 5.5 6.5 7.5 8.5 9.5]
[-5.625 -4.375 -3.125 -1.875 -0.625  0.625  1.875  3.125  4.375  5.625]
cl
[[1. 9.]
 [1. 8.]
 [1. 8.]]
[[-5  5]
 [-5  5]
 [-5  5]]


IndexError: Index 3 is out of bounds for axis 0 with size 3

In [ ]:
sens_count3 = sens_counter(lista_sens_prova3)
chunk_interval3 = np.array([5, 5, 5])

points_count_prova3 = points_counter(chunk_interval3, r_prova3)

termine_noto1, A1, B1 = chunk_matrices(0, chunk_interval3, lista_sens_prova3, r_prova3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto2, A2, B2 = chunk_matrices(1, chunk_interval3, lista_sens_prova3, r_prova3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto3, A3, B3 = chunk_matrices(2, chunk_interval3, lista_sens_prova3, r_prova3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto4, A4, B4 = chunk_matrices(3, chunk_interval3, lista_sens_prova3, r_prova3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto5, A5, B5 = chunk_matrices(4, chunk_interval3, lista_sens_prova3, r_prova3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto6, A6, B6 = chunk_matrices(5, chunk_interval3, lista_sens_prova3, r_prova3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto7, A7, B7 = chunk_matrices(6, chunk_interval3, lista_sens_prova3, r_prova3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto8, A8, B8 = chunk_matrices(7, chunk_interval3, lista_sens_prova3, r_prova3, cube3, interesting_dimensions3, points_count_prova3)

A = sp.sparse.vstack([A1, A2, A3, A4, A5, A6, A7, A8])
B = sp.sparse.vstack([B1, B2, B3, B4, B5, B6, B7, B8])
Mat3 = sp.sparse.hstack([A, B])
termine_noto3 = np.hstack([termine_noto1, termine_noto2, termine_noto3, termine_noto4, termine_noto5, termine_noto6, termine_noto7, termine_noto8]).flatten()

#dovrei controlare che faccia la media
#posso scrivere meglio polynomial
#dovrei scriverli sotto forma di test
#controllo che gestisca 2 sensori => 2 dimensioni LOS

TypeError: 'NoneType' object is not subscriptable

In [ ]:
axes_names3 = cube3.axes_names
l_s_prova3 = data_set_prova3_1, data_set_prova3_2
ch_prova3 = {axes_names3[i]: chunk_interval3[i] for i in range(len(axes_names3))}
axes3 = cube3.axes
masch_ds_prova3 = xr.DataArray(name='masch_ds', data=r_prova3,
                    dims=axes_names3, coords={axes_names3[i]: axes3[i] for i in range(len(axes_names3))})

cube3 = cube3.assign_attrs({"interesting_dimensions" : interesting_dimensions3, "conto_punti" : points_count_prova3})

obj_prova3 = [cube3, masch_ds_prova3]

for el in l_s_prova3:
    obj_prova3.append(el)
    
mis_new3, A3, B3 = map_blocks(chunk_matrices_tagliata, obj_prova3, chunksizes = ch_prova3)

A_comp3 = sp.sparse.vstack(A3)
B_comp3 = sp.sparse.vstack(B3)
Mat_new3 = sp.sparse.hstack([A_comp3, B_comp3])

2025-02-10 11:45:07 [warning  ] No global client found a new one will be created
2025-02-10 11:45:07 [debug    ] <Cluster info: type=LOCAL, n_jobs=22, job_memory=1444500461>
2025-02-10 11:45:08 [debug    ] Selected scatter strategy: once-and-for-all
2025-02-10 11:45:08 [debug    ] Number of blocks to be processed: 8


/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


no valid points in chunk 5


/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


no valid points in chunk 3


/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/pytho

In [ ]:
#10 punti
#simulo_sensore a mano

punti_1 = np.array([[1.2, 0.9, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.4, 5.1, 3.9],
                    [6.1, 6.2, 6.9], [7.1, 6.2, 4.9], [4.8, 8.1, 1.7], 
                    [8.2, 4.9, 0.8], [9, 0.8, 2.3], [7.9, 5.8, 8.3]])
punti_2 = np.array([[1.3, 0.7, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [3.1, 5.1, 3.6],
                    [6.3, 6.3, 6.9], [7.1, 6.2, 4.9], [4.8, 8.4, 1.9], 
                    [8.4, 4.7, 0.8], [9, 0.8, 2.3], [7.9, 5.9, 8.3]])
LOS_1 = np.array([[0.75, 0.13], [0.75, 0.13], [0.74, 0.14], 
                  [0.74, 0.14], [0.74, 0.14], [0.76, 0.14], 
                  [0.76, 0.14], [0.75, 0.12], [0.75, 0.12],
                  [0.75, 0.12]])
LOS_2 = np.array([[0.85, 0.08], [0.86, 0.1], [0.85, 0.1],
                  [0.85, 0.1], [0.85, 0.1], [0.86, 0.11],
                  [0.85, 0.1], [0.85, 0.1], [0.85, 0.1],
                  [0.85, 0.0]])

dim = 2

#normalizzo
L_1 = da.sqrt(da.sum(da.square(LOS_1), axis = 1))
L_2 = da.sqrt(da.sum(da.square(LOS_2), axis = 1))

LOS_1 = LOS_1/(L_1[:, None]*da.ones(dim)[None, :])
LOS_2 = LOS_2/(L_2[:, None]*da.ones(dim)[None, :])

d= np.array([[0.5, 0.5], [1, 1], [0.3, 0.3], [0.4, 0.2], [0.3, 0.7],
             [0.5, 0.5], [1, 1], [0.3, 0.3], [0.4, 0.2], [0.3, 0.7]])

coeff = np.array([1, 1, 2, 2, 3, 3, 4, 4, 5, 5])

d_proiettata_1 = np.zeros((LOS_1.shape[0]))
d_proiettata_2 = np.zeros((LOS_1.shape[0]))

for i in range(LOS_1.shape[0]):
    d_proiettata_1[i] = np.dot(LOS_1[i, :], d[i, :])
    d_proiettata_2[i] = np.dot(LOS_2[i, :], d[i, :])

    
#misure = da.random.random_sample(n_punti)
df1 = pd.DataFrame({'x': punti_1[:, 0], 'y': punti_1[:, 1], 'z': punti_1[:, 2], 
                'LOS_x': LOS_1[:, 0], 'LOS_y': LOS_1[:, 1], 'misure': d_proiettata_1+coeff})

df2 = pd.DataFrame({'x': punti_2[:, 0], 'y': punti_2[:, 1], 'z': punti_2[:, 2], 
                'LOS_x': LOS_2[:, 0], 'LOS_y': LOS_2[:, 1], 'misure': d_proiettata_2+coeff})

ddf1 = dd.from_pandas(df1).repartition(partition_size='10MB')
ddf2 = dd.from_pandas(df2).repartition(partition_size='10MB')

if os.path.isdir('tmp/prova3_1'):
    for file in os.listdir('tmp/prova3_1'):
        os.remove('tmp/prova3_1'+'/'+file)
    os.removedirs('tmp/prova3_1')

if os.path.isdir('tmp/prova3_2'):
    for file in os.listdir('tmp/prova3_2'):
        os.remove('tmp/prova3_2'+'/'+file)
    os.removedirs('tmp/prova3_2')

ddf1.to_csv('tmp/prova3_1', mode='x')
ddf2.to_csv('tmp/prova3_2', mode='x')

cube3 = reticolo(['tmp/prova3_1', 'tmp/prova3_2'], np.array([1, 1, 1]))

data_set_prova3_1=clusterizzazione('tmp/prova3_1', cube3, 1)
data_set_prova3_2=clusterizzazione('tmp/prova3_2', cube3, 1)

lista_sens_prova3 = [data_set_prova3_1, data_set_prova3_2]
interesting_dimensions3 = 3
r3 = get_mask(lista_sens_prova3, interesting_dimensions3) 

sens_count3 = sens_counter(lista_sens_prova3)
chunk_interval3 = np.array([5, 5, 5])

points_count_prova3 = points_counter(chunk_interval3, r3)

termine_noto1, A1, B1 = chunk_matrices(0, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto2, A2, B2 = chunk_matrices(1, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto3, A3, B3 = chunk_matrices(2, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto4, A4, B4 = chunk_matrices(3, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto5, A5, B5 = chunk_matrices(4, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto6, A6, B6 = chunk_matrices(5, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto7, A7, B7 = chunk_matrices(6, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto8, A8, B8 = chunk_matrices(7, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)

A = sp.sparse.vstack([A1, A2, A3, A4, A5, A6, A7, A8])
B = sp.sparse.vstack([B1, B2, B3, B4, B5, B6, B7, B8])
Mat = sp.sparse.hstack([A, B])
termine_noto = np.hstack([termine_noto1, termine_noto2, termine_noto3, termine_noto4, termine_noto5, termine_noto6, termine_noto7, termine_noto8]).flatten()


Informazioni insufficienti: le dimensioni della LOS non sono sufficienti a ricostruire le dimensioni richieste
no valid points in chunk 0
no valid points in chunk 1
no valid points in chunk 2
no valid points in chunk 3
no valid points in chunk 4
no valid points in chunk 5
no valid points in chunk 6
no valid points in chunk 7


provo con polinomi di grado diverso e con più punti nello stesso cubetto per controllare la media

In [ ]:
#10 punti
#simulo_sensore a mano

punti_1 = np.array([[1.2, 0.9, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [2, 4, 2]])
punti_2 = np.array([[1.3, 0.7, 5.8], [0.9, 2.1, 3.0], [2.3, 4.1, 1.8], [2, 4, 2]])
LOS_1 = np.array([[0.75, 0.13], [0.75, 0.13], [0.74, 0.14], 
                  [0.76, 0.18]])
LOS_2 = np.array([[0.85, 0.08], [0.86, 0.1], [0.85, 0.1],
                  [0.82, 0.08]])

dim = 2

#normalizzo
L_1 = da.sqrt(da.sum(da.square(LOS_1), axis = 1))
L_2 = da.sqrt(da.sum(da.square(LOS_2), axis = 1))

LOS_1 = LOS_1/(L_1[:, None]*da.ones(dim)[None, :])
LOS_2 = LOS_2/(L_2[:, None]*da.ones(dim)[None, :])

d= np.array([[0.5, 0.5], [1, 1], [0.3, 0.3], [0.4, 0.2]])

coeff = np.array([1, 1, 2, 2])

d_proiettata_1 = np.zeros((LOS_1.shape[0]))
d_proiettata_2 = np.zeros((LOS_1.shape[0]))

for i in range(LOS_1.shape[0]):
    d_proiettata_1[i] = np.dot(LOS_1[i, :], d[i, :])
    d_proiettata_2[i] = np.dot(LOS_2[i, :], d[i, :])

    
#misure = da.random.random_sample(n_punti)
df1 = pd.DataFrame({'x': punti_1[:, 0], 'y': punti_1[:, 1], 'z': punti_1[:, 2], 
                'LOS_x': LOS_1[:, 0], 'LOS_y': LOS_1[:, 1], 'misure': d_proiettata_1+coeff})

df2 = pd.DataFrame({'x': punti_2[:, 0], 'y': punti_2[:, 1], 'z': punti_2[:, 2], 
                'LOS_x': LOS_2[:, 0], 'LOS_y': LOS_2[:, 1], 'misure': d_proiettata_2+coeff})

ddf1 = dd.from_pandas(df1).repartition(partition_size='10MB')
ddf2 = dd.from_pandas(df2).repartition(partition_size='10MB')

if os.path.isdir('tmp/prova3_1'):
    for file in os.listdir('tmp/prova3_1'):
        os.remove('tmp/prova3_1'+'/'+file)
    os.removedirs('tmp/prova3_1')

if os.path.isdir('tmp/prova3_2'):
    for file in os.listdir('tmp/prova3_2'):
        os.remove('tmp/prova3_2'+'/'+file)
    os.removedirs('tmp/prova3_2')

ddf1.to_csv('tmp/prova3_1', mode='x')
ddf2.to_csv('tmp/prova3_2', mode='x')

cube3 = reticolo(['tmp/prova3_1', 'tmp/prova3_2'], np.array([1, 1, 1]))

data_set_prova3_1=clusterizzazione('tmp/prova3_1', cube3, 1)
data_set_prova3_2=clusterizzazione('tmp/prova3_2', cube3, 0)

lista_sens_prova3 = [data_set_prova3_1, data_set_prova3_2]
interesting_dimensions3 = 2
r3 = get_mask(lista_sens_prova3, interesting_dimensions3) 

sens_count3 = sens_counter(lista_sens_prova3)
chunk_interval3 = np.array([5, 5, 5])

points_count_prova3 = points_counter(chunk_interval3, r3)

termine_noto1, A1, B1 = chunk_matrices(0, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto2, A2, B2 = chunk_matrices(1, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto3, A3, B3 = chunk_matrices(2, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto4, A4, B4 = chunk_matrices(3, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto5, A5, B5 = chunk_matrices(4, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto6, A6, B6 = chunk_matrices(5, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto7, A7, B7 = chunk_matrices(6, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)
termine_noto8, A8, B8 = chunk_matrices(7, chunk_interval3, lista_sens_prova3, r3, cube3, interesting_dimensions3, points_count_prova3)

A = sp.sparse.vstack([A1, A2, A3, A4, A5, A6, A7, A8])
B = sp.sparse.vstack([B1, B2, B3, B4, B5, B6, B7, B8])

Mat = sp.sparse.hstack([A, B])
termine_noto = np.hstack([termine_noto1, termine_noto2, termine_noto3, termine_noto4, termine_noto5, termine_noto6, termine_noto7, termine_noto8]).flatten()

[0.5 1.5 2.5]
[-10.   0.  10.]
(4, 3)
[[[[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [0.98530802 0.17078672]]

  [[       nan        nan]
   [0.98530802 0.17078672]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

  [[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

  [[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]]


 [[[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

  [[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

  [[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

 

/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


[[[[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [0.99560014 0.09370354]]

  [[       nan        nan]
   [0.99330738 0.11550086]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

  [[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

  [[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]]


 [[[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

  [[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

  [[       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]
   [       nan        nan]]

  [[0.99421262 0.10697061]
   [       n

TypeError: 'NoneType' object is not subscriptable

/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/array/core.py:4975: RuntimeWarning: divide by zero encountered in divide
  result = function(*args, **kwargs)
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/array/core.py:4975: RuntimeWarning: invalid value encountered in divide
  result = function(*args, **kwargs)
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/array/core.py:4975: RuntimeWarning: invalid value encountered in divide
  result = function(*args, **kwargs)
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/array/core.py:4975: RuntimeWarning: invalid value encountered in divide
  result = function(*args, **kwargs)
/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWa

In [ ]:

punti5_1=np.array([[1, 0, 0], [0, 0, 1]])
punti5_2=np.array([[1, 0, 0], [0, 0, 1]])
punti5_3=np.array([[1, 0, 0], [0, 0, 1]])
punti5_4=np.array([[1, 0, 0], [0, 0, 1]])
punti5_5=np.array([[1, 0, 0], [0, 0, 1]])
punti5_6=np.array([[1, 0, 0], [0, 0, 1]])
dim = 3

In [ ]:

LOS5_1 = np.array([[1, 1, 1], [1, 1, 1]])
LOS5_2 = np.array([[1, 0, 0], [1, 0, 0]])
LOS5_3 = np.array([[0, 1, 0], [0, 1, 0]])
LOS5_4 = np.array([[0, 0, 1], [0, 0, 1]])
LOS5_5 = np.array([[1, 1, 0], [1, 1, 0]])
LOS5_6 = np.array([[0, 1, 1], [0, 1, 1]])

#normalizzo
L5_1 = da.sqrt(da.sum(da.square(LOS5_1), axis = 1))
L5_2 = da.sqrt(da.sum(da.square(LOS5_2), axis = 1))
L5_3 = da.sqrt(da.sum(da.square(LOS5_3), axis = 1))
L5_4 = da.sqrt(da.sum(da.square(LOS5_4), axis = 1))
L5_5 = da.sqrt(da.sum(da.square(LOS5_5), axis = 1))
L5_6 = da.sqrt(da.sum(da.square(LOS5_6), axis = 1))

LOS5_1 = LOS5_1/(L5_1[:, None]*da.ones(dim)[None, :])
LOS5_2 = LOS5_2/(L5_2[:, None]*da.ones(dim)[None, :])
LOS5_3 = LOS5_3/(L5_3[:, None]*da.ones(dim)[None, :])
LOS5_4 = LOS5_4/(L5_4[:, None]*da.ones(dim)[None, :])
LOS5_5 = LOS5_5/(L5_5[:, None]*da.ones(dim)[None, :])
LOS5_6 = LOS5_6/(L5_6[:, None]*da.ones(dim)[None, :])

In [ ]:
df5_1 = pd.DataFrame({'x': punti5_1[:, 0], 'y': punti5_1[:, 1], 'z': punti5_1[:, 2], 
                'LOS_x': LOS5_1[:, 0], 'LOS_y': LOS5_1[:, 1], 'LOS_z': LOS5_1[:, 2], 
                'misure': np.array([6.19615, 2.73205])})

df5_2 = pd.DataFrame({'x': punti5_2[:, 0], 'y': punti5_2[:, 1], 'z': punti5_2[:, 2], 
                'LOS_x': LOS5_2[:, 0], 'LOS_y': LOS5_2[:, 1], 'LOS_z': LOS5_2[:, 2],
                'misure':np.array([4, 3])})

df5_3 = pd.DataFrame({'x': punti5_3[:, 0], 'y': punti5_3[:, 1], 'z': punti5_3[:, 2], 
                'LOS_x': LOS5_3[:, 0], 'LOS_y': LOS5_3[:, 1], 'LOS_z': LOS5_3[:, 2],
                'misure':np.array([6, 4])})

df5_4 = pd.DataFrame({'x': punti5_4[:, 0], 'y': punti5_4[:, 1], 'z': punti5_4[:, 2], 
                'LOS_x': LOS5_4[:, 0], 'LOS_y': LOS5_4[:, 1], 'LOS_z': LOS5_4[:, 2], 
                'misure': np.array([8, 5])})

df5_5 = pd.DataFrame({'x': punti5_5[:, 0], 'y': punti5_5[:, 1], 'z': punti5_5[:, 2], 
                'LOS_x': LOS5_5[:, 0], 'LOS_y': LOS5_5[:, 1], 'LOS_z': LOS5_5[:, 2],
                'misure':np.array([8.53553, 6.41421])})

df5_6 = pd.DataFrame({'x': punti5_6[:, 0], 'y': punti5_6[:, 1], 'z': punti5_6[:, 2], 
                'LOS_x': LOS5_6[:, 0], 'LOS_y': LOS5_6[:, 1], 'LOS_z': LOS5_6[:, 2],
                'misure':np.array([10.94975, 7.41421])})

ddf5_1 = dd.from_pandas(df5_1).repartition(partition_size='10MB')
ddf5_2 = dd.from_pandas(df5_2).repartition(partition_size='10MB')
ddf5_3 = dd.from_pandas(df5_3).repartition(partition_size='10MB')
ddf5_4 = dd.from_pandas(df5_4).repartition(partition_size='10MB')
ddf5_5 = dd.from_pandas(df5_5).repartition(partition_size='10MB')
ddf5_6 = dd.from_pandas(df5_6).repartition(partition_size='10MB')


if os.path.isdir('tmp/prova5_1'):
    for file in os.listdir('tmp/prova5_1'):
        os.remove('tmp/prova5_1'+'/'+file)
    os.removedirs('tmp/prova5_1')

if os.path.isdir('tmp/prova5_2'):
    for file in os.listdir('tmp/prova5_2'):
        os.remove('tmp/prova5_2'+'/'+file)
    os.removedirs('tmp/prova5_2')

if os.path.isdir('tmp/prova5_3'):
    for file in os.listdir('tmp/prova5_3'):
        os.remove('tmp/prova5_3'+'/'+file)
    os.removedirs('tmp/prova5_3')
    

if os.path.isdir('tmp/prova5_4'):
    for file in os.listdir('tmp/prova5_4'):
        os.remove('tmp/prova5_4'+'/'+file)
    os.removedirs('tmp/prova5_4')

if os.path.isdir('tmp/prova5_5'):
    for file in os.listdir('tmp/prova5_5'):
        os.remove('tmp/prova5_5'+'/'+file)
    os.removedirs('tmp/prova5_5')

if os.path.isdir('tmp/prova5_6'):
    for file in os.listdir('tmp/prova5_6'):
        os.remove('tmp/prova5_6'+'/'+file)
    os.removedirs('tmp/prova5_6')

ddf5_1.to_csv('tmp/prova5_1', mode='x')
ddf5_2.to_csv('tmp/prova5_2', mode='x')
ddf5_3.to_csv('tmp/prova5_3', mode='x')
ddf5_4.to_csv('tmp/prova5_4', mode='x')
ddf5_5.to_csv('tmp/prova5_5', mode='x')
ddf5_6.to_csv('tmp/prova5_6', mode='x')

['/home/debora/rose-l-ard/poi/tmp/prova5_6/0.part']

In [ ]:
cube5 = reticolo(['tmp/prova5_1', 'tmp/prova5_2', 'tmp/prova5_3', 'tmp/prova5_4', 'tmp/prova5_5', 'tmp/prova5_6'], np.array([1, 1, 1]))

data_set_prova5_1=clusterizzazione('tmp/prova5_1', cube5, 0)
data_set_prova5_2=clusterizzazione('tmp/prova5_2', cube5, 0)
data_set_prova5_3=clusterizzazione('tmp/prova5_3', cube5, 0)
data_set_prova5_4=clusterizzazione('tmp/prova5_4', cube5, 0)
data_set_prova5_5=clusterizzazione('tmp/prova5_5', cube5, 0)
data_set_prova5_6=clusterizzazione('tmp/prova5_6', cube5, 0)

lista_sens_prova5 = [data_set_prova5_1, data_set_prova5_2, data_set_prova5_3, data_set_prova5_4, data_set_prova5_5, data_set_prova5_6]

l_s5 = data_set_prova5_1, data_set_prova5_2, data_set_prova5_3, data_set_prova5_4, data_set_prova5_5, data_set_prova5_6

interesting_dimensions5 = 3
r_prova5 = get_mask(lista_sens_prova5, interesting_dimensions5) 

/home/debora/miniconda3/envs/newenv/lib/python3.11/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


In [ ]:
#conto i coefficienti utili per ogni sensore
sens_count5 = sens_counter(lista_sens_prova5)#lo faccio qui o in ogni chunk? indifferente

#intervallo di divisione dei chunk
chunk_interval5 = np.array([10, 10, 10])
#chunk_interval = np.array([5, 5, 5])

#conto i punti in ogni chunk
points_count5 = points_counter(chunk_interval5, r_prova5)

axes_names5 = cube5.attrs["axes_names"]

ch5 = {axes_names5[i]: chunk_interval5[i] for i in range(len(axes_names5))}
axes5 = [cube5.coords[name] for name in axes_names5]
masch_ds5 = xr.DataArray(name='masch_ds', data=r_prova5,
                    dims=axes_names5, coords={axes_names5[i]: axes5[i] for i in range(len(axes_names5))})

cube5 = cube5.assign_attrs({"interesting_dimensions" : interesting_dimensions5, "conto_punti" : points_count5})
obj5 = [cube5, masch_ds5]

for el in l_s5:
    obj5.append(el)

mis_new, A, B = map_blocks(chunk_matrices_tagliata, obj5, chunksizes = ch5)

A_comp = sp.sparse.vstack(A)

B_comp = sp.sparse.vstack(B)
M = np.concatenate(mis_new)
M = np.append(M, np.zeros(A_comp.shape[1],))
Mat_new = sp.sparse.hstack([A_comp, B_comp])
C = sp.sparse.identity((A_comp.shape[1]))*10**(-4)
#C = sp.sparse.identity((A_comp.shape[1]*2))
D = sp.sparse.coo_array((A_comp.shape[1], B_comp.shape[1]))
MM = sp.sparse.hstack([C, D])
Mat_new = sp.sparse.vstack([Mat_new, MM])

print(Mat_new.shape)

2025-03-03 18:34:36 [debug    ] Selected scatter strategy: once-and-for-all
2025-03-03 18:34:36 [warning  ] The chunksize for dimension x is larger than data size along with this dimension. In such a case for this dimension a single chunk of size 10 will be used
2025-03-03 18:34:36 [warning  ] The chunksize for dimension y is larger than data size along with this dimension. In such a case for this dimension a single chunk of size 10 will be used
2025-03-03 18:34:36 [warning  ] The chunksize for dimension z is larger than data size along with this dimension. In such a case for this dimension a single chunk of size 10 will be used
2025-03-03 18:34:36 [debug    ] Number of blocks to be processed: 1


2025-03-03 18:34:36,223 - distributed.worker - ERROR - Compute Failed
Key:       execute-0109b116-f856-495d-b4bc-ae108ff9293d
State:     executing
Function:  execute_task
args:      ((<function map_blocks.<locals>.execute at 0x7f6f2280a8e0>, [<xarray.DataArray 'cube' (x: 2, y: 1, z: 2)> Size: 4B
dask.array<empty_like, shape=(2, 1, 2), dtype=bool, chunksize=(2, 1, 2), chunktype=numpy.ndarray>
Coordinates:
  * x        (x) float64 16B 0.0 1.0
  * y        (y) float64 8B 0.0
  * z        (z) float64 16B 0.0 1.0
Attributes:
    spacing:                 [1 1 1]
    BB:                      dask.array<transpose, shape=(3, 2), dtype=float6...
    BB_rel:                  [[-5  5]\n [-5  5]\n [-5  5]]
    bins:                    [array([-0.5,  0.5,  1.5]), array([-0.5,  0.5]),...
    axes_names:              ['x', 'y', 'z']
    interesting_dimensions:  3
    conto_punti:             [0. 2.], <xarray.DataArray 'masch_ds' (x: 2, y: 1, z: 2)> Size: 32B
dask.array<array, shape=(2, 1, 2), dtype=fl

TypeError: 'builtin_function_or_method' object is not subscriptable

In [ ]:
#print(Mat_new)
print(np.linalg.matrix_rank((Mat_new.transpose()@Mat_new).todense()))

12


In [ ]:
z = sp.sparse.linalg.lsqr(Mat_new.transpose()@Mat_new, Mat_new.transpose()@M)
print(z[0])

[ 1.43248227  2.49223604  1.63451192  2.43248074  4.49223739  4.63451341
 -0.47757427  1.56751849  1.50776328  3.36548733  3.63901495  4.49616057]


In [ ]:
vec = np.array([1., 1., 1., 2., 3., 4., 1., 2., 3., 4., 5., 6.])
y = sp.sparse.linalg.lsqr(Mat_new, M)
z = sp.sparse.linalg.lsqr(Mat_new.transpose()@Mat_new, Mat_new.transpose()@M)
solver = pypardiso.PyPardisoSolver()
print(vec)
#print(Mat_new.transpose()@Mat_new)
x = solver.solve(Mat_new.transpose()@Mat_new, Mat_new.transpose()@M)
print(Mat_new.shape)
#print(Mat_new.transpose()@Mat_new.shape)
print(x)
print(y[0])
MAT = (Mat_new.transpose()@Mat_new).todense()
print(MAT.shape)
print(sp.linalg.expm_cond(MAT))
print(Mat_new@x)
print(M)
#Mat_new_err = Mat_new +(np.random.rand(12, 12))/1000
#a, info = cg(Mat_new_err, M)
#y = sp.sparse.linalg.lsqr(Mat_new_err, M)
#print(a)
#print(y[0])
#print(np.random.rand(12, 12)/100000)
#print(np.linalg.matrix_rank(Mat_new))

[1. 1. 1. 2. 3. 4. 1. 2. 3. 4. 5. 6.]
(18, 12)
[-0.49999914 -1.00000036 -1.50000082  0.49999933  1.00000099  1.50000067
  4.46409981  3.49999991  4.99999969  6.50000008  7.47486971  9.18197983]
[-0.48156185 -1.01421651 -1.49534968  0.51843662  0.98578484  1.50465181
  4.45897739  3.48156261  5.01421584  6.49534894  7.47188491  9.18874332]
(12, 12)
8.629393504184264
[ 2.73204881e+00  6.19615119e+00  3.00000077e+00  3.99999923e+00
  3.99999933e+00  6.00000067e+00  4.99999925e+00  8.00000075e+00
  6.41420989e+00  8.53553011e+00  7.41421204e+00  1.09497480e+01
 -4.99999143e-05 -1.00000036e-04 -1.50000082e-04  4.99999326e-05
  1.00000099e-04  1.50000067e-04]
[ 2.73205  6.19615  3.       4.       4.       6.       5.       8.
  6.41421  8.53553  7.41421 10.94975  0.       0.       0.       0.
  0.       0.     ]


In [ ]:
print(Mat_new.todense())

[[5.77350269e-01 5.77350269e-01 5.77350269e-01 0.00000000e+00
  0.00000000e+00 0.00000000e+00 4.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 5.77350269e-01
  5.77350269e-01 5.77350269e-01 4.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [1.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 4.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 1.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 4.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 1.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  4.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  1

In [ ]:
# ADAM optimizer per il problema Ax = b
def adam_least_squares(A, b, lr=0.00001, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=1000000):
    m, n = A.shape
    x = np.zeros(n)  # Inizializziamo la soluzione a 0
    # Inizializzazione dei momenti
    m_t = np.zeros(n)  # Momento primo (media del gradiente)
    v_t = np.zeros(n)  # Momento secondo (varianza del gradiente)
    for t in range(1, max_iter + 1):
        # Gradiente della funzione di perdita
        grad = A.T @ (A @ x - b)
        # ADAM updates
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        # Correzione del bias
        m_hat = m_t / (1 - beta1 ** t)
        v_hat = v_t / (1 - beta2 ** t)
        # Aggiornamento della soluzione
        x -= lr * m_hat / (np.sqrt(v_hat) + eps)
        # Criterio di arresto (se il gradiente è molto piccolo)
        if np.linalg.norm(grad) < 1e-6:
            break
    return x
# Esempio di utilizzo
#A = np.random.randn(1000, 10)  # Matrice di grandi dimensioni (1000 equazioni, 10 variabili)
#b = np.random.randn(1000)
x_solution = adam_least_squares(Mat_new.transpose()@Mat_new, Mat_new.transpose()@M)
print("Soluzione trovata:", x_solution)
print(Mat_new@x)
print(Mat_new@x_solution)
print(M)

Soluzione trovata: [ 0.42727234  0.83636412  1.02727361 -0.42727233 -0.83636409 -1.02727275
  1.11602493  0.875       1.25        1.62499989  1.8687175   2.29549492]
[ 5.7867573   3.1414427   3.92727233  3.07272767  5.83636408  4.16363592
  7.52727324  5.47272676  8.36839587  6.58134413 10.49977059  7.86418941
  0.42727233  0.83636408  1.02727324 -0.42727233 -0.83636408 -1.02727324]
[ 5.78675727  3.1414427   3.92727233  3.07272766  5.8363641   4.16363589
  7.52727316  5.4727268   8.36839589  6.5813441  10.49977055  7.86418943
  0.42727234  0.83636412  1.02727361 -0.42727233 -0.83636409 -1.02727275]
[ 6.19615  2.73205  4.       3.       6.       4.       8.       5.
  8.53553  6.41421 10.94975  7.41421  0.       0.       0.       0.
  0.       0.     ]


In [ ]:
from scipy.linalg import svd

def solve_svd(A,b,rcond=1e-6):
    # compute svd of A
    U, s, Vh = svd(A)

    # filtering numerical zeroes
    m = (abs(s) / np.max(abs(s))) > rcond
    U, s, Vh = U[:,m], s[m], Vh[m, :]

    # U diag(s) Vh x = b <=> diag(s) Vh x = U.T b = c
    c = U.T @ b
    # diag(s) Vh x = c <=> Vh x = diag(1/s) c = w (trivial inversion of a diagonal matrix)
    w = np.diag(1/s) @ c
    # Vh x = w <=> x = Vh.H w (where .H stands for hermitian = conjugate transpose)
    x = Vh.conj().T @ w
    return x

x=solve_svd(Mat_new.transpose()@Mat_new.todense(), Mat_new.transpose()@M)
print(x)
print(Mat_new.transpose()@Mat_new@x)
print(Mat_new.transpose()@Mat_new@vec)
print(np.linalg.matrix_rank(Mat_new.todense()))
print((Mat_new.transpose()@Mat_new).shape)
print(Mat_new.transpose()@M)

[ 0.42727233  0.83636408  1.02727324 -0.42727233 -0.83636408 -1.02727324
  1.116025    0.875       1.25        1.625       1.8687175   2.295495  ]
[13.61288001 23.35552249 19.31999135  9.11288119 15.35551936 11.81998797
 35.7128     28.         40.         52.         59.79896    73.45584   ]
[ 28.4515367   50.42209945  39.27996383  33.9515367   60.42209945
  49.77996383  59.71281292  76.         112.         148.
 179.79898987 217.45584412]
12
(12, 12)
[13.61288001 23.35552249 19.31999135  9.11288119 15.35551936 11.81998797
 35.7128     28.         40.         52.         59.79896    73.45584   ]


In [ ]:
#print(Mat_new.todense())
plt.figure(figsize= (15, 15))
n = sp.linalg.svdvals((Mat_new.transpose()@Mat_new).todense())
print(n)
#plt.imshow(Mat_new.todense())

[35.53082754 33.54608849 33.10183223 32.         32.         32.
  4.43145391  2.5         2.06854609  0.96671386  0.95391151  0.90062636]


<Figure size 1500x1500 with 0 Axes>